# 04 — Mlp Logistic Regression Stacking / 回归

**Chapter 25 — File 4 of 5 / 第25章 — 第4个文件（共5个）**

---

## Summary / 总结

This script demonstrates **stacked generalization with linear meta model on blobs dataset**.

本脚本演示 **stacked generalization with linear meta model on blobs dataset**。

---
## Step 1 — stacked generalization with linear meta model on blobs dataset

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from keras.models import load_model
from keras.utils import to_categorical
from numpy import dstack

---
## Step 2 — load models from file

In [ ]:
def load_all_models(n_models):
	all_models = list()
	for i in range(n_models):

---
## Step 3 — define filename for this ensemble

In [ ]:
filename = 'models/model_' + str(i + 1) + '.h5'

---
## Step 4 — load model from file

In [ ]:
model = load_model(filename)

---
## Step 5 — add to list of members

In [ ]:
all_models.append(model)
		print('>loaded %s' % filename)
	return all_models

---
## Step 6 — create stacked model input dataset as outputs from the ensemble

In [ ]:
def stacked_dataset(members, inputX):
	stackX = None
	for model in members:

---
## Step 7 — make prediction

In [ ]:
yhat = model.predict(inputX, verbose=0)

---
## Step 8 — stack predictions into [rows, members, probabilities]

In [ ]:
if stackX is None:
			stackX = yhat
		else:
			stackX = dstack((stackX, yhat))

---
## Step 9 — flatten predictions to [rows, members x probabilities]

In [ ]:
stackX = stackX.reshape((stackX.shape[0], stackX.shape[1]*stackX.shape[2]))
	return stackX

---
## Step 10 — fit a model based on the outputs from the ensemble members

In [ ]:
def fit_stacked_model(members, inputX, inputy):

---
## Step 11 — create dataset using ensemble

In [ ]:
stackedX = stacked_dataset(members, inputX)

---
## Step 12 — fit standalone model

In [ ]:
model = LogisticRegression(solver='lbfgs', multi_class='multinomial')
	model.fit(stackedX, inputy)
	return model

---
## Step 13 — make a prediction with the stacked model

In [ ]:
def stacked_prediction(members, model, inputX):

---
## Step 14 — create dataset using ensemble

In [ ]:
stackedX = stacked_dataset(members, inputX)

---
## Step 15 — make a prediction

In [ ]:
yhat = model.predict(stackedX)
	return yhat

---
## Step 16 — generate 2d classification dataset

In [ ]:
X, y = make_blobs(n_samples=1100, centers=3, n_features=2, cluster_std=2, random_state=2)

---
## Step 17 — split into train and test

In [ ]:
n_train = 100
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]

---
## Step 18 — load all models

In [ ]:
n_members = 5
members = load_all_models(n_members)
print('Loaded %d models' % len(members))

---
## Step 19 — evaluate standalone models on test dataset

In [ ]:
for model in members:
	testy_enc = to_categorical(testy)
	_, acc = model.evaluate(testX, testy_enc, verbose=0)
	print('Model Accuracy: %.3f' % acc)

---
## Step 20 — fit stacked model using the ensemble

In [ ]:
model = fit_stacked_model(members, testX, testy)

---
## Step 21 — evaluate model on test set

In [ ]:
yhat = stacked_prediction(members, model, testX)
acc = accuracy_score(testy, yhat)
print('Stacked Test Accuracy: %.3f' % acc)

---
## Learning Notes / 学习笔记

- **概念**: stacked generalization with linear meta model on blobs dataset 是机器学习中的常用技术。  
  *stacked generalization with linear meta model on blobs dataset is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Mlp Logistic Regression Stacking / 回归
# Complete Code / 完整代码
# ===============================

# stacked generalization with linear meta model on blobs dataset
from sklearn.datasets import make_blobs
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from keras.models import load_model
from keras.utils import to_categorical
from numpy import dstack

# load models from file
def load_all_models(n_models):
	all_models = list()
	for i in range(n_models):
		# define filename for this ensemble
		filename = 'models/model_' + str(i + 1) + '.h5'
		# load model from file
		model = load_model(filename)
		# add to list of members
		all_models.append(model)
		print('>loaded %s' % filename)
	return all_models

# create stacked model input dataset as outputs from the ensemble
def stacked_dataset(members, inputX):
	stackX = None
	for model in members:
		# make prediction
		yhat = model.predict(inputX, verbose=0)
		# stack predictions into [rows, members, probabilities]
		if stackX is None:
			stackX = yhat
		else:
			stackX = dstack((stackX, yhat))
	# flatten predictions to [rows, members x probabilities]
	stackX = stackX.reshape((stackX.shape[0], stackX.shape[1]*stackX.shape[2]))
	return stackX

# fit a model based on the outputs from the ensemble members
def fit_stacked_model(members, inputX, inputy):
	# create dataset using ensemble
	stackedX = stacked_dataset(members, inputX)
	# fit standalone model
	model = LogisticRegression(solver='lbfgs', multi_class='multinomial')
	model.fit(stackedX, inputy)
	return model

# make a prediction with the stacked model
def stacked_prediction(members, model, inputX):
	# create dataset using ensemble
	stackedX = stacked_dataset(members, inputX)
	# make a prediction
	yhat = model.predict(stackedX)
	return yhat

# generate 2d classification dataset
X, y = make_blobs(n_samples=1100, centers=3, n_features=2, cluster_std=2, random_state=2)
# split into train and test
n_train = 100
trainX, testX = X[:n_train, :], X[n_train:, :]
trainy, testy = y[:n_train], y[n_train:]
# load all models
n_members = 5
members = load_all_models(n_members)
print('Loaded %d models' % len(members))
# evaluate standalone models on test dataset
for model in members:
	testy_enc = to_categorical(testy)
	_, acc = model.evaluate(testX, testy_enc, verbose=0)
	print('Model Accuracy: %.3f' % acc)
# fit stacked model using the ensemble
model = fit_stacked_model(members, testX, testy)
# evaluate model on test set
yhat = stacked_prediction(members, model, testX)
acc = accuracy_score(testy, yhat)
print('Stacked Test Accuracy: %.3f' % acc)

---

➡️ **Next / 下一步**: File 5 of 5